# 02 — Churn prediction model (optional, leakage-free)

Why this notebook exists: give the retention team a ranked outreach list.
`EstimatedRevenueAtRisk` is built from the churn label, so it stays out of
the feature matrix. Success means high recall on churners, not accuracy.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cleaned = pd.read_csv("../data/processed/telco_customer_churn_cleaned.csv")
print(cleaned.shape)

In [ ]:
# Leakage guard: drop the label, its flag, and every column derived
# from churn status before splitting features from target.
leakage_columns = ["customerID", "Churn", "ChurnFlag", "EstimatedRevenueAtRisk", "RiskSegment", "RecommendedAction", "CustomerProfile"]
features = cleaned.drop(columns=[c for c in leakage_columns if c in cleaned.columns])
target = cleaned["ChurnFlag"]
numeric_features = ["tenure", "MonthlyCharges", "TotalChargesClean"]
categorical_features = [c for c in features.columns if c not in numeric_features]
print("numeric:", numeric_features, "categorical count:", len(categorical_features))

In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)
logistic = Pipeline([("prep", preprocessor), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
logistic.fit(X_train, y_train)
logistic_pred = logistic.predict(X_test)
logistic_proba = logistic.predict_proba(X_test)[:, 1]
print(classification_report(y_test, logistic_pred))
print("ROC-AUC", roc_auc_score(y_test, logistic_proba))
print(confusion_matrix(y_test, logistic_pred))

In [ ]:
forest = Pipeline([("prep", preprocessor), ("clf", RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1))])
forest.fit(X_train, y_train)
forest_pred = forest.predict(X_test)
forest_proba = forest.predict_proba(X_test)[:, 1]
print(classification_report(y_test, forest_pred))
print("ROC-AUC", roc_auc_score(y_test, forest_proba))
print(confusion_matrix(y_test, forest_pred))

## Reading the result

Favour recall for churners over accuracy: a missed at-risk customer is a
missed save. Treat scores as prioritisation, not production readiness — the
sample is fictional and cross-sectional.